In [36]:
import pymrio
import pandas as pd
import os

# DB_PATH = 'D:/Programming/databases/IOT_2024_ixi'
DB_PATH = 'D:/Programming/databases/IOT_2022_ixi'

exio = pymrio.parse_exiobase3(path=DB_PATH)
data_path = DB_PATH


In [37]:
exio.calc_all()

In [38]:
type(exio.extensions)
type(exio)
exio.extensions

['air emissions',
 'energy_use',
 'Satellite Accounts_copy',
 'employment',
 'land',
 'material',
 'nutrients',
 'water']

In [39]:
print(exio.water.DataFrames)
print(exio.air_emissions.DataFrames)
print(exio.DataFrames)

['F', 'F_Y', 'S', 'S_Y', 'M', 'D_cba', 'D_pba', 'D_imp', 'D_exp', 'unit', 'D_cba_reg', 'D_pba_reg', 'D_imp_reg', 'D_exp_reg']
['F', 'F_Y', 'S', 'S_Y', 'M', 'D_cba', 'D_pba', 'D_imp', 'D_exp', 'unit', 'D_cba_reg', 'D_pba_reg', 'D_imp_reg', 'D_exp_reg']
['Z', 'Y', 'x', 'A', 'L', 'unit']


In [40]:
 # --- Explore available sectors (region ×   sector MultiIndex) ---
sectors = exio.Z.index.get_level_values("sector").unique()
regions = exio.Z.index.get_level_values("region").unique()

print(f"Regions ({len(regions)}): {list(regions)}")
print(f"\nTotal unique sectors: {len(sectors)}")
print("\nAll sectors:")
for i, s in enumerate(sectors):
    print(f"  {i:3d}  {s}")


Regions (49): ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'GB', 'US', 'JP', 'CN', 'CA', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'CH', 'TR', 'TW', 'NO', 'ID', 'ZA', 'WA', 'WL', 'WE', 'WF', 'WM']

Total unique sectors: 163

All sectors:
    0  Cultivation of paddy rice
    1  Cultivation of wheat
    2  Cultivation of cereal grains nec
    3  Cultivation of vegetables, fruit, nuts
    4  Cultivation of oil seeds
    5  Cultivation of sugar cane, sugar beet
    6  Cultivation of plant-based fibers
    7  Cultivation of crops nec
    8  Cattle farming
    9  Pigs farming
   10  Poultry farming
   11  Meat animals nec
   12  Animal products nec
   13  Raw milk
   14  Wool, silk-worm cocoons
   15  Manure treatment (conventional), storage and land application
   16  Manure treatment (biogas), storage and land application
   17  Forestry, logging and related service activities (0

In [41]:
# --- Define custom final demand matching exio.Y structure ---
all_sectors = exio.Z.index.get_level_values("sector").unique()
sector_130 = all_sectors[130]
# sector_134 = all_sectors[134]

print(f"Sector 130: {sector_130}")
# print(f"Sector 134: {sector_134}")

# Mirror exio.Y exactly: same row and column MultiIndex, all zeros
y_custom = pd.DataFrame(0.0, index=exio.Y.index, columns=exio.Y.columns)

y_custom.loc[("SE", sector_130), ("SE", "Final consumption expenditure by households")] = 1.0
# y_custom.loc[("SE", sector_134), ("SE", "Final consumption expenditure by households")] = 1.0

# Verify: show only non-zero rows in the household column
col = ("SE", "Final consumption expenditure by households")
print("\ny_custom non-zero entries:")
print(y_custom.loc[y_custom[col] > 0, [col]])


Sector 130: Real estate activities (70)

y_custom non-zero entries:
region                                                                      SE
category                           Final consumption expenditure by households
region sector                                                                 
SE     Real estate activities (70)                                         1.0


In [42]:
# --- Total output required to satisfy custom demand ---
# L [industries × industries] @ y_custom [industries × demand_cols]
# x_custom = exio.L @ y_custom
# print(f"x_custom shape: {x_custom.shape}")

# --- Air emission impacts ---
# S [stressors × industries] @ x_custom [industries × demand_cols] → [stressors × demand_cols]
air_impacts = exio.air_emissions.M @ y_custom

# Collapse to a single series (only one demand column is non-zero)
col = ("SE", "Final consumption expenditure by households")
air_impacts_series = air_impacts[col].sort_values(ascending=False)

print(f"Stressors: {len(air_impacts_series)}")
print("\nTop emitting stressors (kg):")
print(air_impacts_series[air_impacts_series > 0].to_frame("kg"))


Stressors: 421

Top emitting stressors (kg):
                                               kg
stressor                                         
CO2_bio - combustion - air      77657.74857246192
CO2 - combustion - air          68140.23616412187
NOx - combustion - air          251.1713795228349
CO - combustion - air          213.80325536031125
SOx - combustion - air         200.12718485113245
NMVOC - combustion - air        47.24970978399117
TSP - combustion - air         45.246615928695064
PM10 - combustion - air         30.32268968818411
CH4_bio - combustion - air     24.317226761855036
PM2_5 - combustion - air        23.10875854066032
CH4 - combustion - air          8.333596558802197
NH3 - combustion - air         3.4055001432031333
N2O_bio - combustion - air     2.8444305363603477
N2O - combustion - air          2.254953857878317
Cu - combustion - air          0.3828234035198348
As - combustion - air         0.36720080154206314
SO2 - combustion - air        0.13185832533578382
Zn - 

In [43]:
print("Columns match index:", (exio.L.columns == y_custom.index).all())
print("L.columns sample:", exio.L.columns[:2].tolist())
print("y_custom.index sample:", y_custom.index[:2].tolist())


Columns match index: True
L.columns sample: [('AT', 'Cultivation of paddy rice'), ('AT', 'Cultivation of wheat')]
y_custom.index sample: [('AT', 'Cultivation of paddy rice'), ('AT', 'Cultivation of wheat')]


In [44]:
print(exio.air_emissions.M.columns.names)
print(y_custom.index.names)
print(air_impacts_series.isna().sum(), "NaNs out of", len(air_impacts_series))
print("NaN in L:", exio.L.isna().sum().sum())
print("NaN in S:", exio.air_emissions.S.isna().sum().sum())

['region', 'sector']
['region', 'sector']
390 NaNs out of 421
NaN in L: 0
NaN in S: 390


In [45]:
L_clean = exio.L.fillna(0)
S_clean = exio.air_emissions.S.fillna(0)

x_custom = L_clean @ y_custom
air_impacts = S_clean @ x_custom

col = ("SE", "Final consumption expenditure by households")
air_impacts_series = air_impacts[col].sort_values(ascending=False)
print(air_impacts_series[air_impacts_series > 0].to_frame("kg"))


                                                                       kg
stressor                                                                 
CO2_bio - combustion - air                              77657.74857246192
CO2 - combustion - air                                  68140.23616412187
HFC - air                                               3987.292602362485
CO2 - non combustion - Cement production - air          2772.332067878891
CH4 - waste - air                                           557.333143614
...                                                                   ...
PM10 - non combustion - Platinum ores and conce...  5.277560384319335e-09
PCDD/F - non combustion - Steel production: bas... 1.1451341726489116e-09
PM2.5 - non combustion - Platinum ores and conc...  5.958438288158706e-10
PCDD/F - non combustion - Pig iron production, ... 2.2172230539960908e-10
PCDD/F - non combustion - Agglomeration plant -... 1.8869187651520797e-10

[399 rows x 1 columns]


In [46]:
# Sum x_custom across demand columns → Series indexed by (region, sector)
x_vec = x_custom.sum(axis=1)

# D_cba: multiply each column of S_clean by the corresponding x value
D_cba_custom = S_clean.multiply(x_vec, axis=1)

# Total footprint per stressor
footprint_total = D_cba_custom.sum(axis=1).sort_values(ascending=False)
print(footprint_total[footprint_total > 0].to_frame("kg"))


                                                                       kg
stressor                                                                 
CO2_bio - combustion - air                              77657.74857246163
CO2 - combustion - air                                   68140.2361641218
HFC - air                                              3987.2926023624827
CO2 - non combustion - Cement production - air          2772.332067878895
CH4 - waste - air                                       557.3331436139997
...                                                                   ...
PM10 - non combustion - Platinum ores and conce...  5.277560384319334e-09
PCDD/F - non combustion - Steel production: bas...  1.145134172648912e-09
PM2.5 - non combustion - Platinum ores and conc...  5.958438288158706e-10
PCDD/F - non combustion - Pig iron production, ... 2.2172230539960898e-10
PCDD/F - non combustion - Agglomeration plant -... 1.8869187651520784e-10

[399 rows x 1 columns]


In [47]:
# # 1. Top upstream industries — summed across all air emission stressors
# upstream_by_industry = D_cba_custom.sum(axis=0).sort_values(ascending=False)
# print("Top 15 upstream industries (all stressors, kg):")
# print(upstream_by_industry.head(15).to_frame("kg"))

Top 15 upstream industries (all stressors, kg):
                                                                          kg
region sector                                                               
SE     Steam and hot water supply                          80474.82394062151
       Paper                                               4616.598220357094
CN     Production of electricity by coal                   3992.697554141144
SE     Real estate activities (70)                        3726.1793332568172
       Manufacture of wood and of products of wood and...  2180.622762265853
       Manufacture of cement, lime and plaster            1326.6084413000467
       Other land transport                               1188.4522544870654
CN     Manufacture of basic iron and steel and of ferr...  1114.208287371188
DK     Production of electricity by biomass and waste     1092.9492608430414
PL     Production of electricity by coal                  1002.5127693314903
SE     Petroleum Refinery   

In [48]:
# Identify GHG stressors in air_emissions
ghg_keywords = ['CO2', 'CH4', 'N2O', 'SF6', 'HFC', 'PFC', 'NF3']
ghg_stressors = [s for s in exio.air_emissions.S.index
               if any(k in s for k in ghg_keywords)]

print(f"GHG stressors ({len(ghg_stressors)}):")
for s in ghg_stressors:
  print(f"  {s}")


GHG stressors (26):
  CH4 - combustion - air
  CH4_bio - combustion - air
  CO2 - combustion - air
  CO2_bio - combustion - air
  N2O - combustion - air
  N2O_bio - combustion - air
  CH4 - non combustion - Extraction/production of (natural) gas - air
  CH4 - non combustion - Extraction/production of crude oil - air
  CH4 - non combustion - Mining of antracite - air
  CH4 - non combustion - Mining of bituminous coal - air
  CH4 - non combustion - Mining of coking coal - air
  CH4 - non combustion - Mining of lignite (brown coal) - air
  CH4 - non combustion - Mining of sub-bituminous coal - air
  CH4 - non combustion - Oil refinery - air
  CO2 - non combustion - Cement production - air
  CO2 - non combustion - Lime production - air
  SF6 - air
  HFC - air
  PFC - air
  CH4 - agriculture - air
  CO2 - agriculture - peat decay - air
  N2O - agriculture - air
  CH4 - waste - air
  CO2 - waste - biogenic - air
  CO2 - waste - fossil - air
  NF3 - air


In [49]:
A_clean = exio.A.fillna(0)

# numpy for efficient repeated matrix-vector products in the loop
A_np = A_clean.to_numpy()               # (7987 × 7987)
S_np = S_clean.to_numpy()               # (420  × 7987)
y_np = y_custom.sum(axis=1).to_numpy()  # (7987,)

tier_impacts = {}
x_tier = y_np.copy()

for tier in range(10):
  tier_impacts[tier] = S_np @ x_tier  # emissions at this tier (420,)
  x_tier = A_np @ x_tier              # propagate one step upstream

# Compile: stressors × tiers
tier_df = pd.DataFrame(
  tier_impacts,
  index=exio.air_emissions.S.index
)
tier_df.columns.name = "tier"

# Total air emissions per tier
totals = tier_df.sum(axis=0)
cumulative_pct = totals.cumsum() / totals.sum() * 100

summary = pd.DataFrame({"kg": totals, "cumulative_%": cumulative_pct})
print(summary)


                     kg       cumulative_%
tier                                      
0    3653.1943075628715 2.3495117899059905
1     82860.58777674886  55.64038862527705
2     32704.30959356992  76.67380609550443
3     15182.40468043112  86.43820289256226
4     8995.989457561782   92.2238746945398
5     5308.271941394285  95.63783148422212
6     3145.937882553697  97.66110687640972
7    1865.8008655899152  98.86107620797391
8    1109.3493056256657  99.57454202809075
9     661.5334503804645  99.99999999999997


In [50]:
A_clean = exio.A.fillna(0)
A_np = A_clean.to_numpy()
S_np = S_clean.to_numpy()
y_np = y_custom.sum(axis=1).to_numpy()

stressor_tiers = {}
industry_tiers = {}
x_tier = y_np.copy()

for tier in range(10):
  D_tier = S_np * x_tier          # (420 × 7987): stressor × industry at this tier
  stressor_tiers[tier] = D_tier.sum(axis=1)   # collapse industries → per stressor
  industry_tiers[tier] = D_tier.sum(axis=0)   # collapse stressors  → per industry
  x_tier = A_np @ x_tier          # propagate one step upstream

# --- GHG stressors × tiers ---
stressor_df = pd.DataFrame(stressor_tiers, index=exio.air_emissions.S.index).loc[ghg_stressors]
stressor_df.columns.name = "tier"

# --- (region, sector) × tiers ---
industry_df = pd.DataFrame(industry_tiers, index=exio.L.index)
industry_df.columns.name = "tier"

# Top 20 industries by cumulative impact across all tiers
top_industries = industry_df.sum(axis=1).sort_values(ascending=False).head(20).index

print("=== GHG emissions by stressor × tier (kg) ===")
print(stressor_df[stressor_df.sum(axis=1) > 0].to_string())

print("\n=== Top 20 upstream (region, sector) pairs by tier (kg) ===")
print(industry_df.loc[top_industries].to_string())

=== GHG emissions by stressor × tier (kg) ===
tier                                                                                     0                      1                     2                    3                     4                     5                      6                      7                      8                      9
stressor                                                                                                                                                                                                                                                                                            
CH4 - combustion - air                                                 0.10810641868496404       4.80569857049296    1.6653329808109687   0.6960796373809818    0.4209317317118184    0.2531842426819291    0.15246159389655428    0.09181875001570962    0.05531424444581406    0.03336264462237082
CH4_bio - combustion - air                                              1.3

In [51]:
import os
os.makedirs("outputs", exist_ok=True)

# --- Stressor CSV: add total column, drop zero rows, sort by total ---
stressor_out = stressor_df.copy()
stressor_out["total"] = stressor_out.sum(axis=1)
stressor_out = (stressor_out[stressor_out["total"] > 0]
              .sort_values("total", ascending=False))
stressor_out.to_csv("outputs/tier_ghg_stressors.csv")

# --- Industry CSV: sort by total across tiers, drop zero rows ---
industry_total = industry_df.sum(axis=1)
industry_out = industry_df.loc[industry_total[industry_total > 0]
                             .sort_values(ascending=False).index].copy()
industry_out["total"] = industry_out.sum(axis=1)
industry_out.to_csv("outputs/tier_industries.csv")

print(f"tier_ghg_stressors.csv — {stressor_out.shape[0]} stressors × {stressor_df.shape[1]} tiers")
print(f"tier_industries.csv    — {industry_out.shape[0]} industries × {industry_df.shape[1]} tiers")

tier_ghg_stressors.csv — 26 stressors × 10 tiers
tier_industries.csv    — 6107 industries × 10 tiers


In [52]:
industry_total = industry_df.sum(axis=1)
industry_out = (industry_df
              .loc[industry_total[industry_total > 0]
                   .sort_values(ascending=False).index]
              .copy())
industry_out["total"] = industry_out.sum(axis=1)
industry_out = industry_out.reset_index()

industry_out.to_csv("outputs/tier_industries.csv", index=False)
print(f"tier_industries.csv — {len(industry_out)} rows, columns:{industry_out.columns.tolist()}")


tier_industries.csv — 6107 rows, columns:['region', 'sector', 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 'total']
